In [1]:
# import datasets

import pandas as pd
df = pd.read_csv('Tweets.csv')


1 - Cleaning

In [2]:
# display dataset

print(df.tail())

                                                   text  label
1995  i just keep feeling like someone is being unki...      3
1996  im feeling a little cranky negative after this...      3
1997  i feel that i am useful to my people and that ...      1
1998  im feeling more comfortable with derby i feel ...      1
1999  i feel all weird when i have to meet w people ...      4


In [3]:
# remove label column

df = df.drop('label', axis=1)

2 - Pre Processing

In [4]:
# remove punctuations

import string
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

In [5]:
# apply changes

df['text'] = df['text'].apply(remove_punctuation)

In [6]:
# check for any punctiations for the first and last 5 of the train dataset

df

,text
0,im feeling rather rotten so im not very ambiti...
1,im updating my blog because i feel shitty
2,i never make her separate from me because i do...
3,i left with my bouquet of red and yellow tulip...
4,i was feeling a little vain when i did this one
...,...
1995,i just keep feeling like someone is being unki...
1996,im feeling a little cranky negative after this...
1997,i feel that i am useful to my people and that ...
1998,im feeling more comfortable with derby i feel ...


In [7]:
# remove capital letters

def convert_to_lowercase(text):
    return text.lower()

In [8]:
# apply changes

df['text'] = df['text'].apply(convert_to_lowercase)


In [9]:
df

,text
0,im feeling rather rotten so im not very ambiti...
1,im updating my blog because i feel shitty
2,i never make her separate from me because i do...
3,i left with my bouquet of red and yellow tulip...
4,i was feeling a little vain when i did this one
...,...
1995,i just keep feeling like someone is being unki...
1996,im feeling a little cranky negative after this...
1997,i feel that i am useful to my people and that ...
1998,im feeling more comfortable with derby i feel ...


3 - Text Classification Using Dictionary

In [10]:
# classify using dictionary

class EmotionClassifier:
    def __init__(self):
        self.emotion_keywords = {
            'joy': ['joy', 'happy', 'excited', 'glad', 'gassed', 'amazing', 'fire', 'lit', 'insane'],
            'anger': ['anger', 'angry', 'frustrated', 'irritated', 'furious', 'annoyed',],
            'sadness': ['sad', 'depressed', 'unhappy', 'tearful', 'down'],
            'fear': ['fear', 'afraid', 'scared', 'anxious'],
            'surprise': ['surprise', 'amazed', 'astonished', 'shocked', 'stunned', 'crazy'],
            
        }
    
    def classify(self, text):
        for emotion, keywords in self.emotion_keywords.items(): # checks every word in the dictionary
            for keyword in keywords: # loops inside the array of words in each class
                if keyword in text: #checks if keyword is present in the text
                    return emotion
        
        # else classify as:
        return 'Unclassified'

In [11]:
# apply classification

classifier = EmotionClassifier()
df['emotion'] = df['text'].apply(lambda x: classifier.classify(x))

In [12]:
df[['text','emotion']]

,text,emotion
0,im feeling rather rotten so im not very ambiti...,Unclassified
1,im updating my blog because i feel shitty,Unclassified
2,i never make her separate from me because i do...,Unclassified
3,i left with my bouquet of red and yellow tulip...,Unclassified
4,i was feeling a little vain when i did this one,joy
...,...,...
1995,i just keep feeling like someone is being unki...,Unclassified
1996,im feeling a little cranky negative after this...,joy
1997,i feel that i am useful to my people and that ...,Unclassified
1998,im feeling more comfortable with derby i feel ...,Unclassified


4 - Classify using VADER. 

In [13]:
# import vader lexicon that we will use (pretrained model for detecting emotions from prompt)

import nltk
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/hamoaster/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [14]:
# create vader sentiment intensity analyzer

from nltk.sentiment.vader import SentimentIntensityAnalyzer
sid = SentimentIntensityAnalyzer()

In [15]:
# create a function that takes the prompt from the dataset and returns a sentiment score

def sentimentscore(text):
    return sid.polarity_scores(text)

In [16]:
# apply changes and and create a score column for the dataset

df['scores'] = df['text'].apply(sentimentscore)

In [17]:
df[['text','scores']]

,text,scores
0,im feeling rather rotten so im not very ambiti...,"{'neg': 0.39, 'neu': 0.514, 'pos': 0.096, 'com..."
1,im updating my blog because i feel shitty,"{'neg': 0.375, 'neu': 0.625, 'pos': 0.0, 'comp..."
2,i never make her separate from me because i do...,"{'neg': 0.148, 'neu': 0.67, 'pos': 0.182, 'com..."
3,i left with my bouquet of red and yellow tulip...,"{'neg': 0.0, 'neu': 0.817, 'pos': 0.183, 'comp..."
4,i was feeling a little vain when i did this one,"{'neg': 0.251, 'neu': 0.6, 'pos': 0.15, 'compo..."
...,...,...
1995,i just keep feeling like someone is being unki...,"{'neg': 0.144, 'neu': 0.756, 'pos': 0.101, 'co..."
1996,im feeling a little cranky negative after this...,"{'neg': 0.287, 'neu': 0.587, 'pos': 0.126, 'co..."
1997,i feel that i am useful to my people and that ...,"{'neg': 0.0, 'neu': 0.585, 'pos': 0.415, 'comp..."
1998,im feeling more comfortable with derby i feel ...,"{'neg': 0.0, 'neu': 0.733, 'pos': 0.267, 'comp..."


In [18]:
# create a function that classifies emotion using score

def classifyemotion(score):
    if score >= 0.5:
        return 'Happy'
    elif score <= -0.5:
        return 'Unhappy'
    else:
        return 'Neutral'

In [19]:
# apply changes and create a score column for the dataset

df['compound'] = df['scores'].apply(lambda x: x['compound'])
df['predicted_emotion'] = df['compound'].apply(classifyemotion)

In [20]:
df[['text','compound','predicted_emotion']]

,text,compound,predicted_emotion
0,im feeling rather rotten so im not very ambiti...,-0.6778,Unhappy
1,im updating my blog because i feel shitty,-0.5574,Unhappy
2,i never make her separate from me because i do...,-0.0772,Neutral
3,i left with my bouquet of red and yellow tulip...,0.4243,Neutral
4,i was feeling a little vain when i did this one,-0.2516,Neutral
...,...,...,...
1995,i just keep feeling like someone is being unki...,-0.4019,Neutral
1996,im feeling a little cranky negative after this...,-0.4445,Neutral
1997,i feel that i am useful to my people and that ...,0.8176,Happy
1998,im feeling more comfortable with derby i feel ...,0.6240,Happy


5 - Classify Using Transformer "Sentiment-Analysis"

In [21]:
# import transformer model to detect emotions from huggingface

from transformers import pipeline
classifier = pipeline('sentiment-analysis')

2024-05-24 23:18:03.880101: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision af0f99b (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
/Users/hamoaster/anaconda3/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [22]:
# create a function to get sentiment from the transformer i imported

def get_transformer_sentiment(text):
    result = classifier(text)
    return result[0]['label'], result[0]['score']

In [23]:
# apply transformer on the dataset

df['Transformer_Emotion'] = df['text'].apply(lambda x: get_transformer_sentiment(x)[0])
df['Transformer_Score'] = df['text'].apply(lambda x: get_transformer_sentiment(x)[1])

In [24]:
df[['Transformer_Emotion','Transformer_Score']]

,Transformer_Emotion,Transformer_Score
0,NEGATIVE,0.999811
1,NEGATIVE,0.999460
2,POSITIVE,0.999308
3,POSITIVE,0.986825
4,NEGATIVE,0.999649
...,...,...
1995,NEGATIVE,0.995050
1996,NEGATIVE,0.999750
1997,POSITIVE,0.999869
1998,POSITIVE,0.851396
